In [60]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from transformers import BertTokenizer, BertForSequenceClassification, Trainer, TrainingArguments
from torch.utils.data import Dataset, Subset
import torch


In [61]:
df1 = pd.read_csv("data/KyrgyzKabarSport.csv")
df2 = pd.read_csv("data/KyrgyzKabarKyrsyk.csv")
df3 = pd.read_csv("data/KyrgyzKabarDuino.csv")
df4 = pd.read_csv("data/KyrgyzKabarUkukTartibi.csv")
df5 = pd.read_csv("data/KyrgyzKabarPolitika.csv")
df6 = pd.read_csv("data/KyrgyzKabarEAES.csv")

In [4]:
import pandas as pd

# Путь и новая метка
files_and_labels = [
    ("data/KyrgyzKabarSport.csv", "sport"),
    ("data/KyrgyzKabarKyrsyk.csv", "kyrsyk"),
    ("data/KyrgyzKabarDuino.csv", "duino"),
    ("data/KyrgyzKabarUkukTartibi.csv", "ukuktartibi"),
    ("data/KyrgyzKabarPolitika.csv", "politika"),
    ("data/KyrgyzKabarMadaniat.csv", "madaniat")
]

# Обработка всех файлов
for path, new_label in files_and_labels:
    df = pd.read_csv(path)
    df['topic'] = new_label  # заменить метку
    df.to_csv(path, index=False)  # сохранить обратно без индекса


In [62]:
df = pd.concat([df1, df2, df3, df4, df5, df6]).reset_index(drop=True)

In [63]:
print(df.head())
print(df['topic'].unique())

                                               Title  topic
0  Християн Браузман жана Эрбол Атабаев “Абдыш-Ат...  sport
1  Быйыл 20га жакын футболдук объект курулат жана...  sport
2         Дзюдо боюнча өлкө чемпионаты жыйынтыкталды  sport
3  Дзюдо боюнча өлкө чемпионатынын биринчи күнү ж...  sport
4  Бүгүн дзюдо боюнча Кыргызстандын чемпионаты ба...  sport
['sport' 'kyrsyk' 'duino' 'ukuktartibi' 'politika' 'eaes']


In [64]:
# 4. Кодировка меток
label_encoder = LabelEncoder()
df['label'] = label_encoder.fit_transform(df['topic'])
print(dict(zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_))))

{'duino': 0, 'eaes': 1, 'kyrsyk': 2, 'politika': 3, 'sport': 4, 'ukuktartibi': 5}


In [65]:
# 5. Удалим пустые строки (если есть)
df = df.dropna(subset=['Title', 'label']).reset_index(drop=True)

In [66]:
# 6. Загрузка токенизатора BERT
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

In [67]:
# 7. Токенизация текстов
encodings = tokenizer(list(df['Title']), truncation=True, padding=True, max_length=64)


In [68]:
# 8. Кастомный Dataset
class NewsDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(int(self.labels[idx]))  # важно: int!
        return item

    def __len__(self):
        return len(self.labels)

labels = df['label'].tolist()
dataset = NewsDataset(encodings, labels)

In [69]:
# 9. Разделение на train/test
train_indices, test_indices = train_test_split(
    list(range(len(labels))),
    train_size=0.8,
    stratify=labels,
    random_state=42
)

train_dataset = Subset(dataset, train_indices)
test_dataset = Subset(dataset, test_indices)


In [70]:
# 10. Модель
model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=6)

from sklearn.metrics import accuracy_score, precision_recall_fscore_support

def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)  # выбираем класс с наибольшей вероятностью

    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='weighted')
    acc = accuracy_score(labels, preds)

    return {
        'accuracy': acc,
        'precision': precision,
        'recall': recall,
        'f1': f1
    }



Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [71]:
# 11. Параметры обучения
training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=10,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=64,
    warmup_steps=500,
    weight_decay=0.01,
    logging_dir='./logs',
    logging_steps=10,
)

In [73]:
# 12. Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)

In [74]:


# 13. Обучение
trainer.train()


Step,Training Loss
10,1.799400
20,1.780900
30,1.750500
40,1.756100
50,1.735400
60,1.701100
70,1.691800
80,1.664300
90,1.639100
100,1.652500


TrainOutput(global_step=2860, training_loss=0.2734575027611328, metrics={'train_runtime': 683.0123, 'train_samples_per_second': 66.88, 'train_steps_per_second': 4.187, 'total_flos': 1502418082590720.0, 'train_loss': 0.2734575027611328, 'epoch': 10.0})

In [75]:
# Оценка после обучения
results = trainer.evaluate()
print(results)

{'eval_loss': 0.3261379897594452, 'eval_accuracy': 0.9439579684763573, 'eval_precision': 0.9441933769595103, 'eval_recall': 0.9439579684763573, 'eval_f1': 0.9439207405131577, 'eval_runtime': 3.7197, 'eval_samples_per_second': 307.016, 'eval_steps_per_second': 4.839, 'epoch': 10.0}


In [76]:

model_path = "bert-title-classifier2"

# Сохраняем модель и токенизатор
model.save_pretrained(model_path)
tokenizer.save_pretrained(model_path)


('bert-title-classifier2/tokenizer_config.json',
 'bert-title-classifier2/special_tokens_map.json',
 'bert-title-classifier2/vocab.txt',
 'bert-title-classifier2/added_tokens.json')

In [182]:
import torch
from transformers import BertTokenizer, BertForSequenceClassification

# Загрузка модели и токенизатора
model_path = "bert-title-classifier2"
tokenizer = BertTokenizer.from_pretrained(model_path)
model = BertForSequenceClassification.from_pretrained(model_path)
model.eval()  

labels = ["duino", "eaes", "kyrsyk", "politika", "sport", "ukuktartibi"]

def classify_title(title):
    inputs = tokenizer(title, return_tensors="pt", truncation=True, padding=True)
    with torch.no_grad():
        outputs = model(**inputs)
    logits = outputs.logits
    predicted_class = torch.argmax(logits, dim=1).item()
    return labels[predicted_class]

while True:
    text = input("кыргызча Title жазгыла")
    if text.lower() == "exit":
        break
    result = classify_title(text)
    print(f"➡️{text} Этот заголовок относится к категории: **{result}**")


➡️Зеленский Путин менен көзмө көз жолугушууга "даяр" экенин айтты Этот заголовок относится к категории: **duino**
➡️Алмазбек Карасартов Улуттук гвардиянын командачысы болуп дайындалды Этот заголовок относится к категории: **politika**
➡️Газадагы согуш: ортомчулар сунуштаган жаңы формула тууралуу эмне белгилүү? Этот заголовок относится к категории: **eaes**
➡️COVID-19: Бүт Кытай боюнча крематорийде кезек күткөндөр толтура Этот заголовок относится к категории: **duino**


KeyboardInterrupt: Interrupted by user